# African Bat Specimen Records Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² African Bat Specimen dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.azad-7ksp/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll inspect the Croissant metadata for available record sets. All entities, including record sets and fields, are referenced by their `@id` as per Croissant convention.

In [ ]:
from pprint import pprint
# Identify record sets by their @id in the Croissant schema

# The record sets are nested under the 'recordSet' key in the Croissant metadata
# If not present, scan the metadata for entities with '@type': 'cr:RecordSet' or matching schema

croissant = dataset.metadata.to_jsonld()
record_set_ids = []
record_sets = []
for obj in croissant['@graph']:
    if '@type' in obj and (obj['@type'] == 'cr:RecordSet' or \
       (isinstance(obj['@type'], list) and 'cr:RecordSet' in obj['@type'])):
        record_set_ids.append(obj['@id'])
        record_sets.append(obj)

print('Available Record Sets:')
for rs in record_sets:
    print(f"- @id: {rs['@id']}  |  name: {rs.get('name', 'N/A')}")
    # List available fields for each record set
    if 'field' in rs:
        field_ids = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print('  Fields:')
        for fid in field_ids:
            print(f'    - {fid}')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# Extract data from each record set
# For this dataset, we use the main specimen record set. Replace this @id if needed based on overview above.
main_record_set_id = record_set_ids[0]
# You may customize to select other or additional record sets.
dataframes = {}

# Load all records from the main record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
dataframes[main_record_set_id] = df

print(f'>> Loaded {len(df)} records from record set {main_record_set_id}')
print('Available columns (field @id):')
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll pick representative numeric and categorical fields (by their field `@id`, not human names) identified in the schema. Typical numeric fields may include latitude, longitude, or year. We'll walk through filtering, normalizing, and grouping steps.


In [ ]:
# Choose numeric and categorical fields based on their @id
# These should be taken from the dataset's schema. Below are typical guesses for geospatial/specimen data.

# Example field @ids (you might need to replace these if inspection shows different ones):
numeric_field_id = '@latitude'    # Replace with proper field @id from previous output, e.g. 'cr:Latitude' or similar
group_field_id = '@country'       # Replace with actual field @id for country

# Attempt to guess the actual column names; if not, print columns for manual selection
if numeric_field_id not in df.columns or group_field_id not in df.columns:
    print('Please review the following columns and update field ids below:')
    print(df.columns.tolist())
    # For demonstration, let's pick any suitable numeric and groupable field from the columns
    # E.g., guessing 'cr:Latitude' and 'cr:Country' as plausible field ids
    if 'cr:Latitude' in df.columns:
        numeric_field_id = 'cr:Latitude'
    elif 'Latitude' in df.columns:
        numeric_field_id = 'Latitude'
    else:
        numeric_field_id = df.select_dtypes(include=[float, int]).columns[0]  # fallback
    if 'cr:Country' in df.columns:
        group_field_id = 'cr:Country'
    elif 'Country' in df.columns:
        group_field_id = 'Country'
    else:
        group_field_id = df.select_dtypes(include=['object']).columns[0]

# Remove outliers, filter, normalize
try:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype in [int, float] else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else filtered_df[numeric_field_id]
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized", group_field_id]].head())

    # Group by chosen metadata field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
except Exception as e:
    print(f"EDA could not be completed: {e}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group
if group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and inspected the FAIR² African bat specimen dataset with Croissant.
- Identified schema-driven field identifiers and loaded data into pandas DataFrames for analysis.
- Performed exploratory data analysis and visualizations referencing `@id` fields as required for FAIR interoperability.
- Users can adapt the above example by selecting different record sets or fields (by their `@id`) as shown in the metadata inspection section.